In [1]:
import requests
from typing import List
from xml.etree import ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import re
import json
import time
from tqdm import tqdm
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# config for extraction
ENTREZ_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
ENTREZ_ELINK_URL = f"{ENTREZ_URL}/elink.fcgi"
GWAS_MESH_FILTER = '"Genome-Wide Association Study"[MeSH] OR "Genetic Association Studies"[MeSH] OR Polymorphism, Single Nucleotide[MeSH]'
GWAS_TIAB_FILTER = '(GWAS[tiab] OR "genome-wide association"[tiab] OR "polygenic risk"[tiab] OR "polygenic score"[tiab] OR "genetic variant"[tiab] OR "genetic risk"[tiab] OR "SNP"[tiab] OR "single nucleotide polymorphism"[tiab] OR "risk loci"[tiab] OR "genetic association"[tiab] OR "meta-analysis"[tiab] OR "Mendelian randomization"[tiab] OR "whole genome sequencing"[tiab] OR "whole exome sequencing"[tiab] OR "copy number variant"[tiab] OR "heritability"[tiab] OR "gene expression"[tiab] OR "transcriptome"[tiab] OR "eQTL"[tiab] OR "epigenome"[tiab] OR "DNA methylation"[tiab])'
GWAS_FILTER = f'({GWAS_MESH_FILTER}) AND ({GWAS_TIAB_FILTER})'
AD_MESH_FILTER = '"Alzheimer Disease"[MeSH]'
AD_TIAB_FILTER = '"Alzheimer"[tiab] OR "dementia"[tiab] OR "cognitive impairment"[tiab] OR "neurodegeneration"[tiab]'
AD_FILTER = f'({AD_MESH_FILTER}) AND ({AD_TIAB_FILTER})'
ARTICLE_FILTER = '(Journal Article[pt])'
# AND (Meta-Analysis[pt] OR Comparative Study[pt] OR Multicenter Study[pt] OR Evaluation Study[pt] OR Validation Study[pt])
FREE_TEXT_FILTER = '(free full text[sb])'

# filter unnecessary pmids
advp1 = pd.read_csv("test_tables/ADVP_1026_v3p8_extracted.txt", sep = "\t", encoding="cp1252")
advp1_all_pmid = advp1["Pubmed ID"].apply(lambda x: str(int(x)) if not pd.isna(x) else "").unique()

def make_session():
    s = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=2,          # waits 2, 4, 8, 16, 32s between retries
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    return s

session = make_session()

def extract_papers_by_year(year: int) -> List[dict]:
    # 1. Search for PMIDs
    params = {
        "db": "pubmed",
        "term": f"{GWAS_FILTER} AND {AD_FILTER} AND (Humans[MesH]) AND {FREE_TEXT_FILTER}",
        "datetype": "pdat",
        "mindate": f"{year}/01/01",
        "maxdate": f"{year}/12/31",
        "retmode": "json",
        "retmax": 10000,
        "api_key": os.environ.get("ENTREZ_API_KEY", ""),
    }
    r = session.get(f"{ENTREZ_URL}/esearch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    pmids = r.json().get("esearchresult", {}).get("idlist", [])
    pmids_in_advp1 = [
        pmid for pmid in pmids
        if pmid in advp1_all_pmid
    ]
    if len(pmids_in_advp1) > 0:
        print(f"Found {len(pmids_in_advp1)} papers that was in ADVP1")
    pmids = [
        pmid for pmid in pmids
        if pmid not in advp1_all_pmid
    ]
    if not pmids:
        return []
    time.sleep(2)
    # 2. Fetch metadata in batches
    papers = []
    for i in range(0, len(pmids), 10):
        batch_pmids = pmids[i:min(i + 10, len(pmids))]
        batch_pmids = [str(pmid) for pmid in batch_pmids]
        # fetch pubmed id -> pmcid (POST with repeated id= fields for correct 1:1 mapping)
        pmids_to_pmcids = {}
        r = session.post(
            ENTREZ_ELINK_URL,
            data=[
                ("dbfrom", "pubmed"),
                ("db", "pmc"),
                ("linkname", "pubmed_pmc"),
                ("retmode", "xml"),
                ("api_key", os.environ.get("ENTREZ_API_KEY", "")),
                *[("id", pmid) for pmid in batch_pmids],
            ],
            timeout=60,
        )
        r.raise_for_status()
        root = ET.fromstring(r.content)

        for linkset in root.findall("./LinkSet"):
            pmid_elems = linkset.findall("./IdList/Id")
            if not pmid_elems:
                continue
            pmid = pmid_elems[0].text
            pmcids = []
            for db in linkset.findall("./LinkSetDb"):
                dbto = db.find("DbTo")
                if dbto is not None and dbto.text == "pmc":
                    pmcids.extend([x.text for x in db.findall("./Link/Id")])
            if len(pmcids) > 0:
                pmids_to_pmcids[pmid] = f"PMC{pmcids[0]}"

        time.sleep(2)

        removed_pmid = []
        for pmid in pmids_to_pmcids:
            pmcid = pmids_to_pmcids[pmid]
            try:
                has_gwas_table = False
                url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"
                response = requests.get(url)
                if response.status_code == 200:
                    data = response.json()
                    for d in data:
                        doc = d["documents"]
                        for p in doc:
                            passage = p["passages"]
                            for item in passage:
                                if item.get("infons", "").get("type", "").lower() == "table" and "text" in item:
                                    table_str = item["text"]
                                    if re.search(r"rs\d+", table_str) and re.search(r"\d+\.\d+", table_str):
                                        has_gwas_table = True
                                        break
                            if has_gwas_table:
                                break
                        if has_gwas_table:
                            break
                if not has_gwas_table:
                    removed_pmid.append(pmid)
                time.sleep(0.33)
            except Exception as e:
                removed_pmid.append(pmid)

        batch_pmids_to_pmcids = list(pmids_to_pmcids.keys())
        batch_pmids_to_pmcids = [str(pmid) for pmid in batch_pmids_to_pmcids if pmid not in removed_pmid]
        # fetch other metadata
        r = session.get(
            f"{ENTREZ_URL}/efetch.fcgi",
            params={
                "db": "pubmed",
                "id": ",".join(batch_pmids_to_pmcids),
                "rettype": "xml",
                "retmode": "xml",
                "api_key": os.environ.get("ENTREZ_API_KEY", ""),
            },
            timeout=60,
        )
        r.raise_for_status()

        for article in ET.fromstring(r.content).findall(".//PubmedArticle"):
            art = article.find(".//Article")
            title_el = art.find(".//ArticleTitle")
            papers.append({
                "pmid": article.findtext(".//MedlineCitation/PMID", ""),
                "pmcid": pmids_to_pmcids[article.findtext(".//MedlineCitation/PMID", "")],
                "title": "".join(title_el.itertext()) if title_el is not None else "",
                "abstract": " ".join(
                    "".join(el.itertext())
                    for el in art.findall(".//AbstractText")
                ),
                "journal": art.findtext(".//Journal/Title", ""),
                "year": (
                    art.findtext(".//Journal/JournalIssue/PubDate/Year")
                    or art.findtext(".//Journal/JournalIssue/PubDate/MedlineDate", "")[:4]
                ),
                "authors": [
                    f"{a.findtext('LastName', '')}, {a.findtext('ForeName', '')}".strip(", ")
                    for a in art.findall(".//AuthorList/Author")
                    if a.findtext("LastName")
                ],
            })

        time.sleep(2)

    return papers

In [3]:
print(len(advp1_all_pmid))

126


In [4]:
papers = []
for year in range(2009, 2027):
    print(year)
    papers.extend(extract_papers_by_year(year))
    print(f"Total paper found until {year}: {len(papers)}")
    print()

papers = pd.DataFrame(papers)
papers.to_csv("new_gwas_ad_paper.csv", index = False)

2009
Found 4 papers that was in ADVP1
Total paper found until 2009: 7

2010
Found 12 papers that was in ADVP1
Total paper found until 2010: 26

2011
Found 13 papers that was in ADVP1
Total paper found until 2011: 43

2012
Found 12 papers that was in ADVP1
Total paper found until 2012: 71

2013
Found 13 papers that was in ADVP1
Total paper found until 2013: 94

2014
Found 11 papers that was in ADVP1
Total paper found until 2014: 119

2015
Found 8 papers that was in ADVP1
Total paper found until 2015: 148

2016
Found 9 papers that was in ADVP1
Total paper found until 2016: 186

2017
Found 10 papers that was in ADVP1
Total paper found until 2017: 220

2018
Found 9 papers that was in ADVP1
Total paper found until 2018: 255

2019
Found 8 papers that was in ADVP1
Total paper found until 2019: 282

2020
Total paper found until 2020: 316

2021
Total paper found until 2021: 366

2022
Total paper found until 2022: 403

2023
Total paper found until 2023: 439

2024
Total paper found until 2024: 47

In [ ]:
def get_mesh_terms(pmid: int):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed", 
        "id": pmid, 
        "rettype": "xml", 
        "retmode": "xml", 
        "api_key": os.environ.get("ENTREZ_API_KEY", "")
    }
    
    r = requests.get(url, params=params)
    root = ET.fromstring(r.content)
    
    mesh_terms = []
    for heading in root.findall(".//MeshHeading"):
        descriptor = heading.find("DescriptorName").text
        qualifiers = [q.text for q in heading.findall("QualifierName")]
        mesh_terms.append({"descriptor": descriptor, "qualifiers": qualifiers})
    
    return mesh_terms

def get_mesh_terms_batch(batch_pmids: List[int]):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    batch_pmids = [str(int(pmid)) for pmid in batch_pmids]
    params = {
        "db": "pubmed", 
        "id": ",".join(batch_pmids), 
        "rettype": "xml", 
        "retmode": "xml", 
        "api_key": os.environ.get("ENTREZ_API_KEY", "")
    }
    
    r = requests.get(url, params=params)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    
    pmid_to_mesh_terms = {}
    for article in root.findall(".//PubmedArticle"):
        pmid_elem = article.find(".//PMID")
        if pmid_elem is None or pmid_elem.text is None:
            continue

        pmid = pmid_elem.text
        mesh_terms = []
        for heading in article.findall(".//MeshHeading"):
            descriptor_elem = heading.find("DescriptorName")
            descriptor = descriptor_elem.text if descriptor_elem is not None else None

            qualifiers = []
            for q in heading.findall("QualifierName"):
                if q.text:
                    qualifiers.append(q.text)

            if descriptor:
                mesh_terms.append({
                    "descriptor": descriptor,
                    "qualifiers": qualifiers
                })
        
        has_gwas = False
        for term in mesh_terms: 
            if term["descriptor"] == 'Genome-Wide Association Study':
                has_gwas = True
                break
        
        # if not has_gwas:
        #     pmid_to_mesh_terms[pmid] = mesh_terms
        pmid_to_mesh_terms[pmid] = mesh_terms
    
    return pmid_to_mesh_terms

In [ ]:
# for filename in os.listdir("test_tables"):
#     if ".csv" in filename:
#         pmid = int(filename.split("_")[0])
#         print(pmid)
#         curr_mesh_terms = get_mesh_terms(pmid)
#         print(curr_mesh_terms)
#         print([term["descriptor"] for term in curr_mesh_terms])
#         for term in curr_mesh_terms:
#             count_dict[term["descriptor"]] = count_dict.get(term["descriptor"], 0) + 1

# for filename in os.listdir("papers"):
#     if ".pdf" in filename:
#         pmid = int(filename.split("_")[0])
#         print(pmid)
#         curr_mesh_terms = get_mesh_terms(pmid)
#         print(curr_mesh_terms)
#         print([term["descriptor"] for term in curr_mesh_terms])
#         for term in curr_mesh_terms:
#             count_dict[term["descriptor"]] = count_dict.get(term["descriptor"], 0) + 1

count_dict = {}
count_with_mesh_term = 0
pmid_without_gwas_mesh = []

advp1 = pd.read_csv("test_tables/ADVP_1026_v3p8_extracted.txt", sep = "\t", encoding="cp1252")
advp1_all_pmid = advp1["Pubmed ID"].unique()
advp1_all_pmid = advp1_all_pmid[~np.isnan(advp1_all_pmid)]
advp1_all_pmid = advp1_all_pmid.tolist()
for i in tqdm(range(0, len(advp1_all_pmid), 50)):
    batch_pmids = advp1_all_pmid[i: min(i + 50, len(advp1_all_pmid))]
    pmid_to_mesh_terms = get_mesh_terms_batch(batch_pmids)
    for pmid in pmid_to_mesh_terms:
        pmid_without_gwas_mesh.append(pmid)
        if len(pmid_to_mesh_terms[pmid]) > 0:
            count_with_mesh_term += 1
        for term in pmid_to_mesh_terms[pmid]:
            count_dict[term["descriptor"]] = count_dict.get(term["descriptor"], 0) + 1

print(f"{count_with_mesh_term} out of {advp1['Pubmed ID'].nunique()} ADVP1 papers")

In [ ]:
sorted(count_dict.items(), key = lambda x: x[1], reverse = True)

In [ ]:
def fetch_titles_batch(batch_pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    pmid_str = ",".join(map(str, batch_pmids))

    params = {
        "db": "pubmed",
        "id": pmid_str,
        "retmode": "xml",
        "api_key": os.environ.get("ENTREZ_API_KEY", "")
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()

    root = ET.fromstring(r.content)

    results = {}

    for article in root.findall(".//PubmedArticle"):
        pmid_elem = article.find("./MedlineCitation/PMID")
        title_elem = article.find(".//ArticleTitle")

        if pmid_elem is None or pmid_elem.text is None:
            continue

        pmid = pmid_elem.text

        # ArticleTitle can contain nested tags → use itertext()
        if title_elem is not None:
            title = "".join(title_elem.itertext()).strip()
        else:
            title = None

        results[pmid] = title

    return results

pmid_without_gwas_mesh = [int(pmid) for pmid in pmid_without_gwas_mesh]
titles = fetch_titles_batch(pmid_without_gwas_mesh)
titles

In [ ]:
counter = {}
for t in titles:
    for w in titles[t].split(" "):
        counter[w] = counter.get(w, 0) + 1
sorted(counter.items(), key = lambda x: x[1], reverse = True)

In [ ]:
def fetch_article_types_batch(batch_pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(map(str, batch_pmids)),
        "retmode": "xml",
        "api_key": os.environ.get("ENTREZ_API_KEY", ""),
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    root = ET.fromstring(r.content)

    results = {}
    for article in root.findall(".//PubmedArticle"):
        pmid_elem = article.find("./MedlineCitation/PMID")
        if pmid_elem is None or pmid_elem.text is None:
            continue
        pmid = pmid_elem.text
        pub_types = [
            pt.text
            for pt in article.findall(".//PublicationTypeList/PublicationType")
            if pt.text
        ]
        results[pmid] = pub_types
    return results

advp1 = pd.read_csv("test_tables/ADVP_1026_v3p8_extracted.txt", sep = "\t", encoding="cp1252")
advp1_all_pmid = advp1["Pubmed ID"].apply(lambda x: str(int(x)) if not pd.isna(x) else "").unique()
advp1_all_pmid = [pmid for pmid in advp1_all_pmid if pmid]

pmid_to_article_types = {}
for i in tqdm(range(0, len(advp1_all_pmid), 50)):
    batch = advp1_all_pmid[i:i + 50]
    pmid_to_article_types.update(fetch_article_types_batch(batch))
    time.sleep(1)

article_types_df = pd.DataFrame([
    {"pmid": pmid, "article_types": types}
    for pmid, types in pmid_to_article_types.items()
])
article_types_df

In [ ]:
article_types_df.explode("article_types").groupby("article_types", as_index = False)[["pmid"]].count().sort_values("pmid", ascending = False)